# 模型訓練、評估與驗證

## 📌 學習目標

完成本 Notebook 後，你將能夠：

1. 說明訓練集、驗證集、測試集在模型開發流程中的角色。
2. 使用 K-fold 交叉驗證評估模型泛化能力。
3. 透過不同模型複雜度觀察偏差與變異的權衡。
4. 使用分類指標評估模型表現，包括 Accuracy、Precision、Recall、F1-score 與 AUC。
5. 理解早停策略如何避免模型過度擬合訓練資料。

本練習使用輕量的 sklearn 範例資料與模型，模擬實務中的模型訓練、評估與驗證流程。


In [ ]:
# ── 環境設定 ────────────────────────────────────
# 載入本章節所需的 Python 套件，並設定隨機種子，讓每次執行結果可重現。

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, SGDClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)

np.random.seed(42)
print("環境設定完成")


## 核心概念說明

在模型訓練流程中，資料通常會被切分為三種用途：

- **訓練集**：用來讓模型學習參數。
- **驗證集**：用來調整超參數，例如模型複雜度、正規化強度或學習率。
- **測試集**：只在最後使用，用來估計模型面對未知資料時的真實表現。

若資料量有限，可以使用 **K-fold 交叉驗證**。它會將資料分成 K 份，每次拿其中一份當驗證資料，其餘 K-1 份當訓練資料，最後取 K 次結果的平均，使評估更穩定。

模型選擇時常見的核心問題是 **偏差-變異權衡**：

- 偏差高：模型太簡單，無法捕捉資料趨勢，容易欠擬合。
- 變異高：模型太複雜，過度記住訓練資料細節，容易過擬合。
- 最佳模型複雜度通常位於總誤差最低的位置。


In [ ]:
# ── 示範：資料分割與 K-fold 交叉驗證 ────────────────────
# 這段程式碼示範如何將資料切分為訓練集與測試集，並用 K-fold 交叉驗證估計模型的平均表現。

import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X, y = make_classification(
    n_samples=600,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    class_sep=1.2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(model, X, y, cv=kf, scoring="accuracy")

print(f"測試集 Accuracy: {test_acc:.3f}")
print(f"5-fold CV Accuracy: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")


## 偏差-變異權衡

模型複雜度會影響訓練誤差與測試誤差：

- 複雜度太低時，訓練誤差與測試誤差都偏高，代表模型欠擬合。
- 複雜度適中時，測試誤差最低，代表模型具有較佳泛化能力。
- 複雜度太高時，訓練誤差很低，但測試誤差升高，代表模型過擬合。

以下使用多項式迴歸模擬模型複雜度從低到高時的誤差變化。


In [ ]:
# ── 示範：模型複雜度與偏差-變異權衡 ────────────────────────
# 這段程式碼使用不同次方的多項式迴歸模型，觀察訓練誤差與測試誤差如何隨模型複雜度改變。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error

np.random.seed(42)
X = np.linspace(-3, 3, 120).reshape(-1, 1)
y = np.sin(X).ravel() + np.random.normal(0, 0.25, size=X.shape[0])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.35, random_state=42
)

degrees = range(1, 16)
train_errors = []
test_errors = []

for degree in degrees:
    model = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    train_errors.append(mean_squared_error(y_train, train_pred))
    test_errors.append(mean_squared_error(y_test, test_pred))

best_degree = list(degrees)[int(np.argmin(test_errors))]

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_errors, marker="o", label="訓練誤差")
plt.plot(degrees, test_errors, marker="o", label="測試誤差")
plt.axvline(best_degree, color="red", linestyle="--", label=f"最佳複雜度 degree={best_degree}")
plt.xlabel("模型複雜度：多項式次方")
plt.ylabel("Mean Squared Error")
plt.title("模型複雜度與偏差-變異權衡")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"測試誤差最低的多項式次方: {best_degree}")


## 分類評估指標與早停概念

分類模型不能只看 Accuracy。當資料類別不平衡時，模型即使幾乎都預測成多數類，也可能得到看似很高的準確率。因此常搭配以下指標：

- **Precision**：預測為正類的樣本中，有多少真的為正類。
- **Recall**：實際為正類的樣本中，有多少被模型抓出來。
- **F1-score**：Precision 與 Recall 的調和平均，適合需要平衡兩者時使用。
- **ROC AUC**：觀察不同分類門檻下，模型區分正負類的能力。

訓練過程中，也可以使用 **早停策略**。當驗證集表現連續多輪沒有改善時，就停止訓練，避免模型繼續記住訓練資料雜訊。


In [ ]:
# ── 實際應用：分類指標、ROC 曲線與早停模擬 ───────────────────
# 這段程式碼訓練一個二元分類模型，計算混淆矩陣與常見分類指標，並用 SGDClassifier 示範簡化版早停流程。

import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, log_loss

X, y = make_classification(
    n_samples=800,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.65, 0.35],
    class_sep=1.0,
    random_state=7
)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.25, stratify=y_train_full, random_state=7
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, random_state=7)
clf.fit(X_train_s, y_train)

y_pred = clf.predict(X_test_s)
y_score = clf.predict_proba(X_test_s)[:, 1]

print("混淆矩陣:")
print(confusion_matrix(y_test, y_pred))
print("\n分類報告:")
print(classification_report(y_test, y_pred, digits=3))

fpr, tpr, thresholds = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="隨機猜測")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC 曲線")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

sgd = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=0.01,
    alpha=0.0001,
    random_state=7
)

classes = np.unique(y_train)
max_epochs = 80
patience = 6
best_val_loss = np.inf
best_epoch = 0
wait = 0
train_losses = []
val_losses = []

for epoch in range(1, max_epochs + 1):
    sgd.partial_fit(X_train_s, y_train, classes=classes)

    train_prob = sgd.predict_proba(X_train_s)
    val_prob = sgd.predict_proba(X_val_s)
    train_loss = log_loss(y_train, train_prob, labels=classes)
    val_loss = log_loss(y_val, val_prob, labels=classes)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss - 1e-4:
        best_val_loss = val_loss
        best_epoch = epoch
        wait = 0
    else:
        wait += 1

    if wait >= patience:
        print(f"早停觸發：第 {epoch} 輪停止，最佳驗證 loss 出現在第 {best_epoch} 輪")
        break
else:
    print(f"未觸發早停：完成 {max_epochs} 輪，最佳驗證 loss 出現在第 {best_epoch} 輪")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o", label="訓練 loss")
plt.plot(range(1, len(val_losses) + 1), val_losses, marker="o", label="驗證 loss")
plt.axvline(best_epoch, color="red", linestyle="--", label=f"最佳 epoch={best_epoch}")
plt.xlabel("Epoch")
plt.ylabel("Log Loss")
plt.title("早停模擬：訓練與驗證 loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"ROC AUC: {roc_auc:.3f}")
print(f"最佳驗證 loss: {best_val_loss:.3f}")
